# Chapter 8: Selecting an Ultimate Claim Estimate

Code of Chapter 8 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

## 8.2.2 The large claims of the synthetic portfolio

In [ ]:
from gitriangle import synthetic_history, triangle, exposure_by_accident_year
from gidevelop import average_factors, cumulative_factors, develop, latest_diagonal
from giselect import large_claims, capped_triangle

pol, cl, txs = synthetic_history(2020, 2025, seed=100)
big = large_claims(cl, txs, "2025-12-31", threshold=25_000)
big = big[big["accident_year"] >= 2021]
len(big), big.groupby("accident_year").size().tolist()     # 62, [11, 7, 5, 20, 19]

## 8.2.5 The excess load

In [ ]:
Ic = capped_triangle(cl, txs, 25_000, 2021, 2025)
Ix = capped_triangle(cl, txs, 25_000, 2021, 2025, part="excess")
I = triangle(cl, txs, "incurred", 2021, 2025)
dI = develop(I, cumulative_factors(average_factors(I, "volume"), 1.0))
dc = develop(Ic, cumulative_factors(average_factors(Ic, "volume"), 1.0))
load = (dI["ultimate"] - dc["ultimate"]).sum() / dc["ultimate"].sum()   # 0.0922
dc.loc[2025, "ultimate"] * (1 + load)                                    # 3204548

## 8.3.3 The synthetic portfolio

In [ ]:
from giselect import expected_emergence
I24 = triangle(cl, txs, "incurred", 2021, 2024)          # the triangle of the previous review
d24 = develop(I24, cumulative_factors(average_factors(I24, "volume"), 1.0))
ee = expected_emergence(d24["ultimate"], d24["cdf"], dI["cdf"].loc[2021:2024])
actual = dI["latest"].loc[2021:2024] - d24["latest"]
ee.sum().round(0), actual.sum().round(0)                 # 323133, 334223

## 8.4.3 The reserve

In [ ]:
from giselect import selection_summary
P = triangle(cl, txs, "paid", 2021, 2025)
R = triangle(cl, txs, "reported_count", 2021, 2025)
E = exposure_by_accident_year(pol, range(2021, 2026))
dR = develop(R, cumulative_factors(average_factors(R, "volume"), 1.0))
selected = dI["ultimate"].copy()
selected[2025] = dc.loc[2025, "ultimate"] * (1 + load)
s = selection_summary(selected, latest_diagonal(P)["latest"], dI["latest"],
                      E["earned_premium"], E["earned_exposure"], dR["ultimate"])
s["unpaid"].sum().round(0)                                # 2345251

## Exercise 3: Choosing the cap

In [ ]:
# true number of claims above each cap, by accident year (the table above)
ay = cl["accident_date"].dt.year
for cap in (20_000, 25_000, 50_000):
    print(cap, [int(((ay == y) & (cl["ultimate_amount"] > cap)).sum()) for y in range(2021, 2026)])
# 20000 [17, 12, 14, 26, 38]
# 25000 [11, 7, 5, 20, 28]
# 50000 [3, 0, 2, 3, 1]

## 8.7 Python Walkthrough

In [ ]:
from gitriangle import synthetic_history, triangle, exposure_by_accident_year
from gidevelop import average_factors, cumulative_factors, develop, latest_diagonal
from giselect import (large_claims, capped_triangle, expected_emergence,
                      selection_summary)

# Setup: the data and objects built in the sections of this chapter
pol, cl, txs = synthetic_history(2020, 2025, seed=100)
Ic = capped_triangle(cl, txs, 25_000, 2021, 2025)
Ix = capped_triangle(cl, txs, 25_000, 2021, 2025, part="excess")
I = triangle(cl, txs, "incurred", 2021, 2025)
dI = develop(I, cumulative_factors(average_factors(I, "volume"), 1.0))
dc = develop(Ic, cumulative_factors(average_factors(Ic, "volume"), 1.0))
load = (dI["ultimate"] - dc["ultimate"]).sum() / dc["ultimate"].sum()
I24 = triangle(cl, txs, "incurred", 2021, 2024)
d24 = develop(I24, cumulative_factors(average_factors(I24, "volume"), 1.0))
ee = expected_emergence(d24["ultimate"], d24["cdf"], dI["cdf"].loc[2021:2024])
actual = dI["latest"].loc[2021:2024] - d24["latest"]
P = triangle(cl, txs, "paid", 2021, 2025)
R = triangle(cl, txs, "reported_count", 2021, 2025)
E = exposure_by_accident_year(pol, range(2021, 2026))
dR = develop(R, cumulative_factors(average_factors(R, "volume"), 1.0))
selected = dI["ultimate"].copy()
selected[2025] = dc.loc[2025, "ultimate"] * (1 + load)
s = selection_summary(selected, latest_diagonal(P)["latest"], dI["latest"],
                      E["earned_premium"], E["earned_exposure"], dR["ultimate"])

# Section 8.2: large claims and the capped approach
Ix[12].round(0).tolist()               # [230101, 29427, 242211, 196780, 235209]
average_factors(Ic, "volume").round(4).tolist()          # [1.1166, 1.0071, 1.0, 1.0021]

# Section 8.3: roll-forward of the 2024 review
(actual - ee).round(0).tolist()        # [4294, 19408, -58928, 46317]

# Section 8.4: the selection and its ratios
s[["loss_ratio", "pure_premium", "severity", "ibnr_to_case"]].round(4)